In [1]:
import os
import plaid
from plaid.api import plaid_api
from dotenv import load_dotenv

# Load variables from .env
load_dotenv()

# Access variables using os.getenv
PLAID_CLIENT_ID = os.getenv('PLAID_CLIENT_ID')
PLAID_ENV = os.getenv('PLAID_ENV', 'sandbox') # Default to sandbox if not set

# Logic to pick the correct secret based on the environment
if PLAID_ENV == 'production':
    host = plaid.Environment.Production
    secret = os.getenv('PLAID_PRODUCTION_SECRET')
else:
    host = plaid.Environment.Sandbox
    secret = os.getenv('PLAID_SANDBOX_SECRET')

# Configuration
configuration = plaid.Configuration(
    host=host,
    api_key={
        'clientId': PLAID_CLIENT_ID,
        'secret': secret,
    }
)

api_client = plaid.ApiClient(configuration)
client = plaid_api.PlaidApi(api_client)

print(f"Plaid Client initialized in {PLAID_ENV} mode.")

Plaid Client initialized in sandbox mode.


In [2]:
from plaid.model.link_token_create_request import LinkTokenCreateRequest
from plaid.model.link_token_create_request_user import LinkTokenCreateRequestUser
from plaid.model.products import Products
from plaid.model.country_code import CountryCode

def get_link_token():
    request = LinkTokenCreateRequest(
        user=LinkTokenCreateRequestUser(client_user_id='unique-user-id-123'),
        client_name="My Cool App",
        products=[Products('transactions')],
        country_codes=[CountryCode('US')],
        language='en'
    )
    
    response = client.link_token_create(request)
    return response['link_token']

link_token = get_link_token()
print(f"Your Link Token: {link_token}")

Your Link Token: link-sandbox-1b903c78-2acd-42bb-b8d9-0ac435c43c24


In [3]:
from plaid.model.item_public_token_exchange_request import ItemPublicTokenExchangeRequest

def exchange_token(public_token):
    exchange_request = ItemPublicTokenExchangeRequest(
        public_token=public_token
    )
    exchange_response = client.item_public_token_exchange(exchange_request)
    
    # Save these to your database!
    access_token = exchange_response['access_token']
    item_id = exchange_response['item_id']
    return access_token

access_token = exchange_token('link-sandbox-d383a78a-c25f-4bf4-901c-156eb52039d9')
print(access_token)

ApiException: Status Code: 400
Reason: Bad Request
HTTP response headers: HTTPHeaderDict({'Server': 'nginx', 'Date': 'Sun, 01 Feb 2026 17:43:54 GMT', 'Content-Type': 'application/json; charset=utf-8', 'Content-Length': '336', 'Connection': 'keep-alive', 'plaid-version': '2020-09-14', 'x-envoy-upstream-service-time': '48', 'x-envoy-decorator-operation': 'default.svc-apiv2:8080/*'})
HTTP response body: {
  "display_message": null,
  "documentation_url": "https://plaid.com/docs/?ref=error#invalid-input-errors",
  "error_code": "INVALID_PUBLIC_TOKEN",
  "error_message": "provided token is the wrong type. expected \"public\", got \"link\"",
  "error_type": "INVALID_INPUT",
  "request_id": "4LnG4mUNi5Lm7pw",
  "suggested_action": null
}


In [4]:
from plaid.model.transactions_get_request import TransactionsGetRequest
from datetime import date

def get_transactions(access_token):
    request = TransactionsGetRequest(
        access_token=access_token,
        start_date=date(2024, 1, 1),
        end_date=date(2024, 2, 1)
    )
    response = client.transactions_get(request)
    return response['transactions']

In [4]:
import os
import time

import plaid
from plaid.api import plaid_api
from plaid.model.products import Products
from plaid.model.item_public_token_exchange_request import ItemPublicTokenExchangeRequest
from plaid.model.sandbox_public_token_create_request import SandboxPublicTokenCreateRequest
from plaid.model.transactions_get_request import TransactionsGetRequest
from datetime import date, timedelta
from dotenv import load_dotenv

load_dotenv()

# 1. Initialize Plaid Client
configuration = plaid.Configuration(
    host=plaid.Environment.Sandbox,
    api_key={
        'clientId': os.getenv('PLAID_CLIENT_ID'),
        'secret': os.getenv('PLAID_SANDBOX_SECRET'),
    }
)
api_client = plaid.ApiClient(configuration)
client = plaid_api.PlaidApi(api_client)

def run_sandbox_test(max_retries=5):
    # retries = 0
    
    # while retries < max_retries:
    try:
        # 2. Generate a Sandbox Public Token (Bypasses the Frontend UI)
        # 'ins_109' is the ID for "First Platypus Bank" (The standard test bank)
        sandbox_req = SandboxPublicTokenCreateRequest(
            institution_id='ins_5',
            initial_products=[Products('transactions')]
        )
        sandbox_res = client.sandbox_public_token_create(sandbox_req)
        public_token = sandbox_res['public_token']
        print(f"Generated Public Token: {public_token}")

        # 3. Exchange Public Token for Access Token
        exchange_req = ItemPublicTokenExchangeRequest(public_token=public_token)
        exchange_res = client.item_public_token_exchange(exchange_req)
        access_token = exchange_res['access_token']
        print(f"Exchanged Access Token: {access_token}")

        time.sleep(30)

        # 4. Use Access Token to Fetch Transactions
        # Fetching the last 30 days of data
        start_date = date.today() - timedelta(days=30)
        end_date = date.today()
        
        trans_req = TransactionsGetRequest(
            access_token=access_token,
            start_date=start_date,
            end_date=end_date
        )
        trans_res = client.transactions_get(trans_req)
        
        print(f"\nSuccessfully fetched {len(trans_res['transactions'])} transactions!")
        for txn in trans_res['transactions'][:3]: # Print first 3
            print(f"- {txn['date']}: {txn['name']} (${txn['amount']})")

        return trans_res

    except plaid.ApiException as e:
        print(f"Plaid API Error: {e.body}")


trans_res = run_sandbox_test()

Generated Public Token: public-sandbox-12686ae7-29fd-469f-8772-d8a83d849e2e
Exchanged Access Token: access-sandbox-bbefadcc-e24c-4173-9936-dbea0f52e791

Successfully fetched 16 transactions!
- 2026-01-29: United Airlines ($500.0)
- 2026-01-27: Uber 072515 SF**POOL** ($6.33)
- 2026-01-24: Tectra Inc ($500.0)


In [5]:
trans_res2 = run_sandbox_test()

Generated Public Token: public-sandbox-182266f4-98a4-4efd-93d5-d0c35f163784
Exchanged Access Token: access-sandbox-86318183-5c5d-43c2-8806-e91c410d8635

Successfully fetched 16 transactions!
- 2026-01-29: United Airlines ($500.0)
- 2026-01-27: Uber 072515 SF**POOL** ($6.33)
- 2026-01-24: Tectra Inc ($500.0)


In [31]:
trans_res.to_dict().keys()

dict_keys(['accounts', 'transactions', 'total_transactions', 'item', 'request_id'])

In [7]:
set([d['account_id'] for d in trans_res['accounts']]) & set([d['account_id'] for d in trans_res2['accounts']])

set()

['yeD7XbkLRNu9JgRprKvwULqEwz3b4KuprWevW',
 '9A5zawRQj1iNXW5qZKlPSKvb8yV5rkIMlVaJE',
 'vP3NwRgZJdiE6wJ73WjvHLv1QMzngWuVonA7l',
 'R5ZKa1wvAdFQyL4rZXJoIrAyljEnK7u5zX31w',
 '68Lq3Bnj4QuB7G981g5aC4ma57Dv1eunrbLAz',
 'X5QdewbvR7Fb5kJm3oBGco9mWgPkRwi85PqlJ',
 'D54eLwnAGbFEgnvPAGN4H79mweWQkliGzExpG',
 'V5w3jgZvmPFyzvpnZrXqhag6mkGj5MsN6peAA',
 'wjxPG5RM1aFG8BMQ3q7vHN815bDqdniKgqjdD',
 '5gJlBVnPaetRjQMk9GLdCwzakdKNnBtjx3MLa',
 'J5QpqbwvWlFDaWNMbJZyUyzV4jZMn3cwboqNV',
 'kqpl6mM3o9f6jRkenaKvCZgJ5GwyX4cAxzKRM']

In [33]:
[(d['account_id'], d['amount'], d['date']) for d in trans_res['transactions']]

[('BeM8RZmB8wCWQbD4RjaPiKrjw6aLwrh4WpW9Q', 500.0, datetime.date(2026, 1, 29)),
 ('K8KBndm7BRIanoV1dqMDtmvzWbl5WviRZ9Zlz', 6.33, datetime.date(2026, 1, 27)),
 ('BeM8RZmB8wCWQbD4RjaPiKrjw6aLwrh4WpW9Q', 500.0, datetime.date(2026, 1, 24)),
 ('BeM8RZmB8wCWQbD4RjaPiKrjw6aLwrh4WpW9Q', 2078.5, datetime.date(2026, 1, 23)),
 ('BeM8RZmB8wCWQbD4RjaPiKrjw6aLwrh4WpW9Q', 500.0, datetime.date(2026, 1, 23)),
 ('BeM8RZmB8wCWQbD4RjaPiKrjw6aLwrh4WpW9Q', 500.0, datetime.date(2026, 1, 23)),
 ('r8GQJDZMQqIQq6V75Zazu6LMq8WJqLc7PAPrj', 25.0, datetime.date(2026, 1, 14)),
 ('K8KBndm7BRIanoV1dqMDtmvzWbl5WviRZ9Zlz', 5.4, datetime.date(2026, 1, 14)),
 ('3lraKbNEaPHwZq3891xkuMKadGnQdKuZrjr7e', 5850.0, datetime.date(2026, 1, 13)),
 ('zyG8mrbW8JI8o5QXAyZPskNmoEZloNSl7y7Ap', 1000.0, datetime.date(2026, 1, 13)),
 ('BeM8RZmB8wCWQbD4RjaPiKrjw6aLwrh4WpW9Q', 78.5, datetime.date(2026, 1, 12)),
 ('K8KBndm7BRIanoV1dqMDtmvzWbl5WviRZ9Zlz', -500.0, datetime.date(2026, 1, 12)),
 ('K8KBndm7BRIanoV1dqMDtmvzWbl5WviRZ9Zlz', 12.0, date

In [42]:
print(trans_res['accounts'][0])
print()
print(trans_res['transactions'][0])

{'account_id': 'K8KBndm7BRIanoV1dqMDtmvzWbl5WviRZ9Zlz',
 'balances': {'available': 100.0,
              'current': 110.0,
              'iso_currency_code': 'USD',
              'limit': None,
              'unofficial_currency_code': None},
 'holder_category': 'personal',
 'mask': '0000',
 'name': 'Plaid Checking',
 'official_name': 'Plaid Gold Standard 0% Interest Checking',
 'subtype': 'checking',
 'type': 'depository'}

{'account_id': 'BeM8RZmB8wCWQbD4RjaPiKrjw6aLwrh4WpW9Q',
 'account_owner': None,
 'amount': 500.0,
 'authorized_date': None,
 'authorized_datetime': None,
 'category': None,
 'category_id': None,
 'check_number': None,
 'counterparties': [],
 'date': datetime.date(2026, 1, 29),
 'datetime': None,
 'iso_currency_code': 'USD',
 'location': {'address': None,
              'city': None,
              'country': None,
              'lat': None,
              'lon': None,
              'postal_code': None,
              'region': None,
              'store_number': None},


In [35]:
trans_res['total_transactions']

16

In [38]:
trans_res['item']

{'available_products': ['assets',
                        'auth',
                        'balance',
                        'identity',
                        'identity_match',
                        'income_verification',
                        'investments',
                        'investments_auth',
                        'liabilities',
                        'recurring_transactions',
                        'signal',
                        'statements',
                        'transfer'],
 'billed_products': ['transactions'],
 'consent_expiration_time': None,
 'error': None,
 'institution_id': 'ins_5',
 'institution_name': 'Citibank Online',
 'item_id': 'bdGlxXqKlphlMJV1zQPKTqV48wr3yVhVzLyvN',
 'products': ['transactions'],
 'update_type': 'background',
 'webhook': ''}

In [37]:
trans_res['request_id']

'ERLUvxzKWiUM3r3'